# Moduł 12: NLP — przetwarzanie języka naturalnego i embeddingi

NLP to dziedzina AI zajmująca się **rozumieniem i generowaniem** języka naturalnego.

## Spis treści
1. Tokenizacja
2. Prawo Zipfa
3. Reprezentacje tekstu (Bag of Words, TF-IDF)
4. Word embeddings (Word2Vec, GloVe)
5. Operacje na wektorach słów
6. **Zadania**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 1. Tokenizacja

**Tokenizacja** to podział tekstu na mniejsze jednostki (tokeny): słowa, znaki, sub-słowa.

### Typy tokenizacji

| Typ | Przykład | Rozmiar słownika | Zastosowanie |
|-----|---------|-------------------|-------------|
| **Word-level** | "Ala ma" → ["Ala", "ma"] | 50k-200k | Klasyczny NLP |
| **Char-level** | "Ala" → ["A", "l", "a"] | ~100 | Rzadko samodzielnie |
| **BPE (Byte Pair Encoding)** | "playing" → ["play", "##ing"] | 30k-50k | GPT, RoBERTa |
| **WordPiece** | "unhappy" → ["un", "##happy"] | 30k | BERT |
| **SentencePiece** | Bez pre-tokenizacji | 30k-50k | T5, mBERT |

### Algorytm BPE (Byte Pair Encoding)

1. Zacznij od słownika znaków
2. Znajdź najczęściej występującą parę sąsiednich tokenów
3. Połącz tę parę w nowy token
4. Powtarzaj do osiągnięcia pożądanego rozmiaru słownika

**Zaleta:** Balans między word-level (semantyka) a char-level (brak OOV). Rzadkie słowa rozkładane na sub-tokeny.

### Specjalne tokeny
- `[CLS]` — token klasyfikacyjny (BERT)
- `[SEP]` — separator zdań
- `[PAD]` — padding do jednakowej długości
- `[UNK]` — nieznane słowo (OOV)
- `<|endoftext|>` — koniec tekstu (GPT)

In [ ]:
# Prosta tokenizacja
tekst = "Sztuczna inteligencja to fascynujący temat. AI zmienia świat."

# Na słowa (naiwna)
tokeny = tekst.lower().split()
print(f"Tokeny: {tokeny}")
print(f"Liczba tokenów: {len(tokeny)}")

# Budowanie słownika
vocab = sorted(set(tokeny))
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}
print(f"\nSłownik ({len(vocab)} słów): {word2idx}")

## 2. Prawo Zipfa

**Prawo Zipfa:** W dowolnym tekście, częstość $f$ słowa na pozycji $r$ w rankingu spełnia:

$$f(r) \propto \frac{1}{r^\alpha}, \quad \alpha \approx 1$$

Czyli najczęstsze słowo pojawia się ~2× częściej niż drugie, ~3× częściej niż trzecie itd.

**Konsekwencje dla NLP:**
- **Długi ogon**: ~50% słów w korpusie pojawia się tylko raz (hapax legomena)
- Mała grupa słów (stop words) dominuje: "the", "a", "is"
- Na wykresie log-log to **linia prosta** z nachyleniem $-\alpha$

$$\log f = -\alpha \log r + \text{const}$$

**Implikacje praktyczne:**
- Potrzeba dużych korpusów, aby rzadkie słowa miały wystarczające reprezentacje
- Sub-word tokenization (BPE) radzi sobie z rzadkimi słowami

In [ ]:
# Demonstracja Prawa Zipfa
# Symulujemy długi tekst
np.random.seed(42)
# Rozkład Zipfa: prawdopodobieństwo proporcjonalne do 1/r
vocab_size = 1000
ranks = np.arange(1, vocab_size + 1)
probs = 1 / ranks
probs /= probs.sum()

# "Generujemy" tekst wg rozkładu Zipfa
text_indices = np.random.choice(vocab_size, size=100000, p=probs)
counts = Counter(text_indices)
freqs = sorted(counts.values(), reverse=True)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(freqs[:100])
plt.xlabel("Rank")
plt.ylabel("Częstość")
plt.title("Prawo Zipfa (liniowa skala)")

plt.subplot(1, 2, 2)
plt.loglog(range(1, len(freqs)+1), freqs)
plt.xlabel("Rank (log)")
plt.ylabel("Częstość (log)")
plt.title("Prawo Zipfa (log-log) → prosta!")

plt.tight_layout()
plt.show()

## 3. Reprezentacje tekstu

### One-Hot Encoding

Każde słowo = wektor z jedną jedynką, reszta zera:

$$\mathbf{e}_i \in \{0, 1\}^{|V|}, \quad (\mathbf{e}_i)_j = \begin{cases}1 & j = i \\ 0 & j \neq i\end{cases}$$

```
"kot" → [1, 0, 0, 0, 0]
"pies" → [0, 1, 0, 0, 0]
"ryba" → [0, 0, 1, 0, 0]
```

**Problem:** Wszystkie słowa są jednakowo "odległe": $\cos(\mathbf{e}_i, \mathbf{e}_j) = 0$ dla $i \neq j$. Brak semantyki!

### Bag of Words (BoW)

Dokument = wektor częstości słów (ignoruje kolejność):

$$\text{BoW}(d) = [c(w_1, d), c(w_2, d), \ldots, c(w_{|V|}, d)]$$

gdzie $c(w, d)$ = liczba wystąpień słowa $w$ w dokumencie $d$.

### TF-IDF (Term Frequency — Inverse Document Frequency)

Ważona częstość: słowa **rzadkie** w korpusie, ale **częste** w dokumencie → wysokie TF-IDF:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \cdot \text{IDF}(t, D)$$

**Term Frequency** — jak często słowo $t$ pojawia się w dokumencie $d$:
$$\text{TF}(t, d) = \frac{c(t, d)}{\sum_{t'} c(t', d)}$$

**Inverse Document Frequency** — jak rzadkie jest słowo w korpusie $D$:
$$\text{IDF}(t, D) = \log\frac{|D|}{|\{d \in D: t \in d\}|}$$

**Intuicja:** Słowa pojawiające się wszędzie ("the", "a") mają niski IDF → niski TF-IDF. Słowa specyficzne dla dokumentu → wysoki TF-IDF.

### Cosine Similarity

Miara podobieństwa między wektorami:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_i a_i b_i}{\sqrt{\sum_i a_i^2} \cdot \sqrt{\sum_i b_i^2}}$$

- $\cos = 1$ → identyczne kierunki (bardzo podobne)
- $\cos = 0$ → ortogonalne (brak powiązania)
- $\cos = -1$ → przeciwne kierunki

In [ ]:
# One-Hot encoding
words = ["kot", "pies", "ryba", "kot", "ptak"]
vocab = sorted(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}

def one_hot(word, vocab_size, word2idx):
 vec = np.zeros(vocab_size)
 vec[word2idx[word]] = 1
 return vec

for w in vocab:
 print(f"{w:6s} → {one_hot(w, len(vocab), word2idx)}")

# Problem: cosine similarity między dowolnymi dwoma = 0!
v_kot = one_hot("kot", len(vocab), word2idx)
v_pies = one_hot("pies", len(vocab), word2idx)
cos_sim = np.dot(v_kot, v_pies) / (np.linalg.norm(v_kot) * np.linalg.norm(v_pies))
print(f"\nCosine sim (kot, pies) = {cos_sim} ← brak semantyki!")

## 4. Word Embeddings

**Word embeddings** to gęste wektory (np. 300D) nauczone z ogromnych korpusów tekstowych, gdzie podobne słowa mają podobne wektory.

### Hipoteza dystrybucyjna (Firth, 1957)

> "Poznasz słowo po towarzystwie, które mu się towarzyszy"

Słowa występujące w **podobnych kontekstach** → **podobne wektory**!

### Word2Vec — Skip-gram (Mikolov et al., 2013)

**Cel:** Dany jest słowo centralne $w_c$, przewiduj słowa kontekstowe $w_o$:

$$\max \sum_{(w_c, w_o) \in D} \log P(w_o | w_c)$$

$$P(w_o | w_c) = \frac{\exp(\mathbf{u}_{w_o}^T \mathbf{v}_{w_c})}{\sum_{w \in V} \exp(\mathbf{u}_w^T \mathbf{v}_{w_c})}$$

gdzie $\mathbf{v}_{w_c}$ — embedding słowa centralnego, $\mathbf{u}_{w_o}$ — embedding kontekstowy.

**Problem:** Mianownik sumuje po całym słowniku ($|V| \sim 10^5$) → drogie!

**Rozwiązanie:** **Negative sampling** — zamiast pełnego softmax, ucz się rozróżniać prawdziwe pary od losowych:

$$\mathcal{L} = -\log \sigma(\mathbf{u}_{w_o}^T \mathbf{v}_{w_c}) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{w_k}^T \mathbf{v}_{w_c})$$

gdzie $w_k$ to $K$ negatywnych (losowych) próbek.

### Word2Vec — CBOW (Continuous Bag of Words)

Odwrotnie: dany kontekst → przewiduj słowo centralne:

$$P(w_c | w_{c-m}, \ldots, w_{c+m}) = \text{softmax}\left(\mathbf{u}_{w_c}^T \bar{\mathbf{v}}\right)$$

gdzie $\bar{\mathbf{v}} = \frac{1}{2m}\sum_{j=-m, j\neq 0}^{m} \mathbf{v}_{w_{c+j}}$ — średnia embeddingów kontekstu.

### GloVe (Global Vectors, Pennington et al., 2014)

Optymalizuje tak, by iloczyn skalarny = log koincydencji:

$$\mathcal{L} = \sum_{i,j} f(X_{ij})\left(\mathbf{w}_i^T \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2$$

gdzie $X_{ij}$ = liczba ko-wystąpień słów $i, j$ w oknie, $f$ to funkcja wagowa (ogranicza wpływ bardzo częstych par).

### PMI (Pointwise Mutual Information)

Miara siły asocjacji między słowami:

$$\text{PMI}(w_1, w_2) = \log \frac{P(w_1, w_2)}{P(w_1) P(w_2)}$$

- PMI > 0 → słowa ko-występują **częściej** niż losowo
- Levy & Goldberg (2014) pokazali, że Word2Vec implicitnie faktoryzuje macierz PMI

### Właściwości embeddingów
- **Semantyka:** $\cos(\text{king}, \text{queen}) > \cos(\text{king}, \text{car})$
- **Analogie:** $\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$
- **Klastry:** Podobne słowa grupują się w przestrzeni

In [ ]:
# Symulacja word embeddings (w praktyce ładowałbyś GloVe/Word2Vec)
# Tu pokazuję koncepcję z małymi wektorami

# Wyobraź sobie 3D embeddings nauczone z danych:
embeddings = {
 "king": np.array([0.8, 0.6, 0.9]),
 "queen": np.array([0.7, 0.65, 0.85]),
 "man": np.array([0.9, 0.1, 0.5]),
 "woman": np.array([0.8, 0.15, 0.45]),
 "cat": np.array([0.1, 0.8, 0.2]),
 "dog": np.array([0.15, 0.75, 0.25]),
 "car": np.array([0.3, 0.2, 0.1]),
 "truck": np.array([0.35, 0.18, 0.12]),
}

def cosine_sim(a, b):
 return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Porównaj podobieństwa
pairs = [("king", "queen"), ("king", "car"), ("cat", "dog"), ("cat", "truck")]
for w1, w2 in pairs:
 sim = cosine_sim(embeddings[w1], embeddings[w2])
 print(f"sim({w1:6s}, {w2:6s}) = {sim:.3f}")

## 5. Analogie wektorowe

Najbardziej spektakularna właściwość embeddingów:

$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

### Interpretacja geometryczna

Wektor $\vec{\text{king}} - \vec{\text{man}}$ koduje **koncept "królewskości"** niezależny od płci. Dodanie $\vec{\text{woman}}$ przenosi ten koncept na kobietę → queen.

Formalnie, szukamy:
$$w^* = \arg\max_{w \in V \setminus \{a, b, c\}} \cos\left(\mathbf{v}_w, \; \mathbf{v}_b - \mathbf{v}_a + \mathbf{v}_c\right)$$

### Inne przykłady analogii
- `Paris - France + Germany ≈ Berlin` (stolice)
- `walking - walk + swim ≈ swimming` (odmiana)
- `bigger - big + small ≈ smaller` (stopniowanie)

### Ograniczenia embeddingów statycznych

1. **Jedno znaczenie per słowo** — "bank" (finansowy vs rzeka) ma jeden wektor
2. **Brak kontekstu** — każde wystąpienie słowa ma tę samą reprezentację
3. **Bias** — embeddingi dziedziczą stereotypy z danych treningowych

**Rozwiązanie:** Kontekstualne embeddingi (ELMo, BERT) → ten sam słowo, różne wektory w zależności od kontekstu!

In [ ]:
# Analogia: king - man + woman = ?
query = embeddings["king"] - embeddings["man"] + embeddings["woman"]

# Szukamy najbliższego słowa
best_word, best_sim = None, -1
for word, vec in embeddings.items():
 if word in ["king", "man", "woman"]:
 continue
 sim = cosine_sim(query, vec)
 print(f" {word:8s}: {sim:.3f}")
 if sim > best_sim:
 best_sim = sim
 best_word = word

print(f"\nking - man + woman = {best_word} (sim={best_sim:.3f})")

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Tokenizer i BoW (łatwe)

Zbuduj prosty system Bag-of-Words:
1. Napisz tokenizer (lowercase, usuń interpunkcję, split)
2. Zbuduj słownik z korpusu
3. Zamień dokumenty na wektory BoW
4. Oblicz cosine similarity między dokumentami

Korpus:
```python
docs = [
 "Kot siedzi na macie i śpi.",
 "Pies biega po parku z piłką.",
 "Kot i pies bawią się razem.",
 "Samochód jedzie szybko autostradą."
]
```

In [ ]:
# Zadanie 1
# TWÓJ KOD TUTAJ

### Zadanie 2: TF-IDF od zera (średnie)

Zaimplementuj TF-IDF bez użycia sklearn:
- $\text{TF}(t, d) = \frac{\text{count}(t, d)}{|d|}$
- $\text{IDF}(t) = \log\frac{N}{\text{DF}(t)}$
- $\text{TF-IDF} = \text{TF} \cdot \text{IDF}$

Wykorzystaj ten sam korpus co w Zadaniu 1. Które słowa mają najwyższe TF-IDF w każdym dokumencie?

In [ ]:
# Zadanie 2: TF-IDF
# TWÓJ KOD TUTAJ

### Zadanie 3: Skip-gram — intuicja (trudne)

Zaimplementuj uproszczony Skip-gram w PyTorch:
1. Weź zdanie, stwórz pary (słowo centralne, słowo kontekstowe) z oknem = 2
2. Model: `nn.Embedding(vocab_size, embed_dim)` → `nn.Linear(embed_dim, vocab_size)`
3. Loss: CrossEntropyLoss
4. Trenuj na małym korpusie
5. Wizualizuj nauczone embeddingi 2D (PCA)

Korpus: "the cat sat on the mat the dog sat on the rug"

In [ ]:
# Zadanie 3: Skip-gram
import torch
import torch.nn as nn
# TWÓJ KOD TUTAJ

### Zadanie 4: Weryfikacja prawa Zipfa (średnie)

1. Weź dowolny tekst po polsku (możesz użyć wielokrotnie powtórzonego paragrafu lub wkleić fragment książki)
2. Tokenizuj i policz częstości słów
3. Narysuj wykres rank vs frequency (log-log)
4. Sprawdź, czy jest zbliżony do prostej

In [ ]:
# Zadanie 4: Prawo Zipfa na prawdziwym tekście
# TWÓJ KOD TUTAJ

### Zadanie 5: TF-IDF od zera (srednie)

Zaimplementuj TF-IDF bez uzycia sklearn:

$$\text{TF}(t, d) = \frac{\text{count}(t, d)}{|d|}$$
$$\text{IDF}(t) = \log\frac{N}{1 + \text{df}(t)}$$
$$\text{TF-IDF}(t, d) = \text{TF}(t,d) \cdot \text{IDF}(t)$$

1. Napisz funkcje `compute_tfidf(corpus)` -> macierz TF-IDF
2. Przetestuj na 5 krotkich zdaniach
3. Porownaj wynik z `TfidfVectorizer` ze sklearn

In [ ]:
# Zadanie 5: TF-IDF od zera
import numpy as np
from collections import Counter

corpus = [
 "kot siedzi na macie",
 "pies biega po parku",
 "kot i pies sa przyjaciolmi",
 "maca lezy na podlodze",
 "park jest duzy i zielony"
]

def compute_tfidf(corpus):
 # TWOJ KOD TUTAJ
 pass

# tfidf_matrix = compute_tfidf(corpus)
# print(tfidf_matrix.shape)

### Zadanie 6: TF-IDF od podstaw
Zaimplementuj TF-IDF **bez sklearn** na zbiorze 5 dokumentów po polsku.
1. Oblicz TF (term frequency) dla każdego dokumentu
2. Oblicz IDF (inverse document frequency) dla każdego terminu
3. Oblicz macierz TF-IDF
4. Znajdź 3 najbardziej charakterystyczne słowa dla każdego dokumentu
5. Porównaj wynik z `TfidfVectorizer` ze sklearn

In [ ]:
# TWOJ KOD TUTAJ


### Zadanie 7: Nearest Neighbors w przestrzeni embeddingów
Używając gotowych embeddingów (np. z gensim `glove-wiki-gigaword-50`):
1. Dla 10 wybranych słów znajdź po 5 najbliższych sąsiadów (cosine similarity)
2. Zwizualizuj wybrane słowa w 2D (t-SNE lub PCA)
3. Sprawdź analogię: king - man + woman = ?
4. Oblicz średnie podobieństwo wewnątrz grupy tematycznej (np. zwierzęta vs kolory)

In [ ]:
# TWOJ KOD TUTAJ


---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
NLP i embeddingi pojawiają się w wielu zadaniach OAI:

- **I OAI:** Zagadki (retrieval z TF-IDF, MRR@20), analiza zależnościowa (HerBERT, structural probes)
- **I OAI, Finał:** Szyfry (analiza częstości, n-gramy, łamanie szyfrów)
- **II OAI, Etap 2:** Ekstrakcja źródeł (embedding retrieval, 768-dim, SGPT-125M, cosine similarity)
- **II OAI, Finał:** Stylizacja tłumaczeń (MarianMT fine-tuning, BLEU)
- **III OAI:** Zmiany semantyczne (embeddingi diachroniczne, Procrustes alignment, balanced accuracy)

### Ćwiczenie OAI-11A: Retrieval z embeddingów

Stwórz prostą bazę "dokumentów" (zdania), oblicz ich embeddingi (TF-IDF lub sentence embeddings), i zaimplementuj wyszukiwanie najbliższych sąsiadów.

### Ćwiczenie OAI-11B: Wykrywanie zmian semantycznych

Dane: dwa zestawy embeddingów słów z różnych epok. Zadanie: wyrównaj przestrzenie (Procrustes alignment) i wykryj słowa, których znaczenie się zmieniło.

### Ćwiczenie OAI-11C: Analiza n-gramów (przygotowanie do szyfrów)

Zaimplementuj analizę częstości liter i bigramów w tekście polskim. Porównaj z rozkładem oczekiwanym.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# === OAI-10A: Retrieval z embeddingów TF-IDF ===
print("=== Embedding Retrieval (wzorzec: Ekstrakcja źródeł, II OAI etap 2) ===")

# Mini-baza dokumentów
documents = [
 "Sieci neuronowe uczą się reprezentacji danych wielowymiarowych.",
 "Konwolucyjne sieci neuronowe świetnie radzą sobie z obrazami.",
 "Drzewa decyzyjne to popularne modele klasyfikacji.",
 "Transfer learning pozwala wykorzystać pretrenowane modele.",
 "Embeddingi słów reprezentują semantykę w przestrzeni wektorowej.",
 "Gradient descent minimalizuje funkcję kosztu iteracyjnie.",
 "Random Forest łączy wiele drzew decyzyjnych w ensemble.",
 "Atencja pozwala modelowi skupić się na ważnych częściach danych.",
]

# TF-IDF embeddingi
vectorizer = TfidfVectorizer()
doc_embeddings = vectorizer.fit_transform(documents).toarray()

def retrieve_top_k(query, k=3):
 """Wyszukaj k najbardziej podobnych dokumentów do zapytania."""
 query_emb = vectorizer.transform([query]).toarray()
 similarities = cosine_similarity(query_emb, doc_embeddings)[0]
 top_indices = np.argsort(similarities)[::-1][:k]

 print(f"\n Zapytanie: '{query}'")
 for rank, idx in enumerate(top_indices, 1):
 print(f" {rank}. (sim={similarities[idx]:.3f}) {documents[idx]}")
 return top_indices

retrieve_top_k("Jak działają sieci konwolucyjne?", k=3)
retrieve_top_k("Ensemble drzew", k=2)

# === OAI-10B: Procrustes Alignment ===
print("\n\n=== Procrustes Alignment (wzorzec: Zmiany semantyczne, III OAI) ===")

def orthogonal_procrustes(A, B):
 """
 Znajdź macierz rotacji R minimalizującą ||A @ R - B||.
 Używane do wyrównywania embeddingów z różnych epok.
 """
 # Normalizacja L2
 A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
 B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)

 # SVD na M = B^T @ A
 M = B_norm.T @ A_norm
 U, _, Vt = np.linalg.svd(M)
 R = U @ Vt
 return R

# Symulacja: dwa zestawy embeddingów
np.random.seed(42)
dim = 50
n_words = 100

# Epocha 1: losowe embeddingi
W1 = np.random.randn(n_words, dim)

# Epocha 2: obrócone + kilka słów ze zmienioną semantyką
true_rotation = np.linalg.qr(np.random.randn(dim, dim))[0]
W2 = W1 @ true_rotation # Obrót
# Zmień znaczenie 10 słów
changed_indices = list(range(10))
W2[changed_indices] += 2.0 * np.random.randn(10, dim)

# Wyrównaj przestrzenie
R = orthogonal_procrustes(W1, W2)
W1_aligned = W1 @ R

# Mierz zmianę semantyczną = 1 - cosine_similarity
def semantic_distance(w1, w2):
 sim = np.dot(w1, w2) / (np.linalg.norm(w1) * np.linalg.norm(w2) + 1e-10)
 return 1 - sim

distances = [semantic_distance(W1_aligned[i], W2[i]) for i in range(n_words)]
detected_changed = np.argsort(distances)[-15:] # Top 15 najbardziej zmienionych

# Ewaluacja
correctly_detected = len(set(detected_changed) & set(changed_indices))
print(f" Wykryto poprawnie: {correctly_detected}/10 zmienionych słów")
print(f" W top-15 podejrzanych: {correctly_detected} trafień")

# === OAI-10C: Analiza częstości (n-gramy) ===
print("\n\n=== Analiza częstości liter (wzorzec: Szyfry, I OAI finał) ===")

tekst = """Sieci neuronowe to modele matematyczne inspirowane działaniem mózgu.
Składają się z warstw neuronów połączonych wagami, które są uczone na danych.
Głębokie uczenie pozwala na automatyczne odkrywanie cech w danych."""

# Częstość liter
litery = [c.lower() for c in tekst if c.isalpha()]
freq = Counter(litery)
total = len(litery)
print(" Najczęstsze litery:")
for letter, count in freq.most_common(10):
 print(f" '{letter}': {count:3d} ({100*count/total:.1f}%)")

# Bigramy
bigramy = [tekst[i:i+2].lower() for i in range(len(tekst)-1) 
 if tekst[i].isalpha() and tekst[i+1].isalpha()]
bigram_freq = Counter(bigramy)
print("\n Najczęstsze bigramy:")
for bigram, count in bigram_freq.most_common(8):
 print(f" '{bigram}': {count}")

print("\n Na I OAI trzeba było złamać szyfr podstawieniowy korzystając z analizy częstości!")